# Docling Funding Metadata Extraction

In [1]:
%reload_ext sql
%sql duckdb:///:memory:
%config SqlMagic.displaylimit = 0
%config SqlMagic.autopandas = False

The 'toml' package isn't installed. To load settings from pyproject.toml or ~/.jupysql/config, install with: pip install toml

Connecting to 'duckdb:///:memory:'

## Create views

In [2]:
%%sql

CREATE VIEW docling AS
SELECT *
FROM read_parquet('../data/output/docling-results/*.parquet');

CREATE VIEW arxiv_tex_extract AS
SELECT *
FROM read_parquet('../data/output/arxiv-tex-extract-results/*.parquet');

Running query in 'duckdb:///:memory:'

Count


## Overview of results

In [3]:
%%sql

SELECT
    arxiv_id,
    status,
    file_type,
    num_tex_files,
    markdown_length,
    CASE
      WHEN LENGTH(markdown) > 100 THEN CONCAT(SUBSTR(markdown, 1, 100), '...')
      ELSE markdown
    END AS markdown,
    entry_name,
    outer_tar
FROM (
    SELECT
        arxiv_id,
        status,
        file_type,
        num_tex_files,
        markdown_length,
        markdown,
        entry_name,
        outer_tar,
        ROW_NUMBER() OVER (
            PARTITION BY status
            ORDER BY arxiv_id
        ) AS rn
    FROM docling
) t
WHERE rn <= 3
ORDER BY status, arxiv_id;

Running query in 'duckdb:///:memory:'

arxiv_id,status,file_type,num_tex_files,markdown_length,markdown,entry_name,outer_tar
1903.12237,failure,tex,1,None,None,1903.12237.tar.gz,None
2105.09891,failure,tex,1,None,None,2105.09891.tar.gz,None
2310.03985,failure,tex,2,None,None,2310.03985.tar.gz,None
1305.2229,fallback,tex,1,451737,"\documentclass[reqno,usenames]{amsart}\usepackage{tikz}\usepackage{amsfonts}\usepackage{bbm} \...",1305.2229.tar.gz,None
1405.2458,fallback,tex,2,37466,%************************************************************************%* ...,1405.2458.tar.gz,None
1612.04719,fallback,tex,1,222972,"\documentclass[11pt,twoside]{amsart}\usepackage{amsmath}\usepackage{amsfonts}\usepackage{amssymb...",1612.04719.tar.gz,None
0704.0331,skipped,unknown,0,None,None,0704.0331.tar.gz,None
0706.0552,skipped,unknown,0,None,None,0706.0552.tar.gz,None
0710.4739,skipped,unknown,0,None,None,0710.4739.tar.gz,None
0704.0734,success,tex,1,4840,"# GRO J1655-40: from ASCA and XMM-Newton ObservationsXiao-LingZhang1, ShuangNanZhang2,3,4, Glori...",0704.0734.tar.gz,None


## Results Status

In [4]:
%%sql

SELECT status, COUNT(*) AS count
FROM docling
GROUP BY status
ORDER BY count DESC;

Running query in 'duckdb:///:memory:'

status,count
success,2457
skipped,237
fallback,6
failure,5


## Detected File Type

In [5]:
%%sql
    
SELECT file_type, COUNT(*) AS count
FROM docling
GROUP BY file_type
ORDER BY count DESC;

Running query in 'duckdb:///:memory:'

file_type,count
tex,2468
unknown,233
postscript,4


## Number of Tex Files Per Archive

In [6]:
%%sql
    
SELECT num_tex_files, COUNT(*) AS count
FROM docling
GROUP BY num_tex_files
ORDER BY count DESC;

Running query in 'duckdb:///:memory:'

num_tex_files,count
1,1079
2,821
0,237
3,115
4,66
9,40
10,40
8,39
12,37
11,30


In [7]:
#

In [8]:
%%sql

SELECT
    arxiv_id,
    status,
    error_message,
    num_tex_files,
    text_length,
    CASE
      WHEN LENGTH(text) > 100 THEN CONCAT(SUBSTR(text, 1, 100), '...')
      ELSE text
    END AS text,
    stage_timings_us,
    total_time_us,
    peak_memory_bytes,
    outer_tar,
    file_type,
    entry_name,
    shard_id
FROM (
    SELECT
        arxiv_id,
        status,
        error_message,
        num_tex_files,
        text_length,
        text,
        stage_timings_us,
        total_time_us,
        peak_memory_bytes,
        outer_tar,
        file_type,
        entry_name,
        shard_id,
        ROW_NUMBER() OVER (
            PARTITION BY status
            ORDER BY arxiv_id
        ) AS rn
    FROM arxiv_tex_extract
) t
WHERE rn <= 3
ORDER BY status, arxiv_id;

Running query in 'duckdb:///:memory:'

arxiv_id,status,error_message,num_tex_files,text_length,text,stage_timings_us,total_time_us,peak_memory_bytes,outer_tar,file_type,entry_name,shard_id
1209.3999,archive_error,entry 1209.3999.tar.gz raw size (129824822 bytes) exceeds 100MB limit,None,None,None,None,None,None,arxiv-train-test-tars.tar,None,None,arxiv-train-test-tars
2206.02478,archive_error,entry 2206.02478.tar.gz raw size (167915718 bytes) exceeds 100MB limit,None,None,None,None,None,None,arxiv-train-test-tars.tar,None,None,arxiv-train-test-tars
2306.11698,archive_error,entry 2306.11698.tar.gz raw size (118365094 bytes) exceeds 100MB limit,None,None,None,None,None,None,arxiv-train-test-tars.tar,None,None,arxiv-train-test-tars
0704.0331,empty,None,0,None,None,None,None,None,arxiv-train-test-tars.tar,unknown,0704.0331.tar.gz,arxiv-train-test-tars
0706.0552,empty,None,0,None,None,None,None,None,arxiv-train-test-tars.tar,unknown,0706.0552.tar.gz,arxiv-train-test-tars
0710.4739,empty,None,0,None,None,None,None,None,arxiv-train-test-tars.tar,unknown,0710.4739.tar.gz,arxiv-train-test-tars
0704.0734,ok,None,1,4556,"# GRO J1655-40: from ASCA and XMM-Newton ObservationsXiao-Ling Zhang1, Shuang Nan Zhang2,3,4, Glor...","{""remove_comments"":5,""normalize_shorthands"":1,""expand_macros"":0,""expand_parametric"":0,""convert_structure"":47,""convert_references"":39,""convert_formatting"":298,""convert_environments"":366,""strip_pre_diacritic"":130,""convert_diacritics"":37,""convert_symbols"":35,""cleanup"":239}",1197,None,arxiv-train-test-tars.tar,tex,0704.0734.tar.gz,arxiv-train-test-tars
0704.2216,ok,None,1,68961,# Maximally Sparse Polynomials have Solid AmoebasMounir NisseUniversité Pierre et Marie Curie-Pa...,"{""remove_comments"":90,""normalize_shorthands"":37,""expand_macros"":152,""expand_parametric"":1,""convert_structure"":938,""convert_references"":486,""convert_formatting"":6166,""convert_environments"":8365,""strip_pre_diacritic"":2219,""convert_diacritics"":715,""convert_symbols"":769,""cleanup"":3946}",23884,None,arxiv-train-test-tars.tar,tex,0704.2216.tar.gz,arxiv-train-test-tars
0704.2272,ok,None,1,25819,"# THE CONNECTION BETWEEN STAR-FORMING GALAXIES, AGN HOST GALAXIES AND EARLY-TYPE GALAXIES IN THE SDS...","{""remove_comments"":25,""normalize_shorthands"":4,""expand_macros"":19,""expand_parametric"":0,""convert_structure"":264,""convert_references"":154,""convert_formatting"":1655,""convert_environments"":1813,""strip_pre_diacritic"":677,""convert_diacritics"":157,""convert_symbols"":157,""cleanup"":1350}",6275,None,arxiv-train-test-tars.tar,tex,0704.2272.tar.gz,arxiv-train-test-tars


In [9]:
%%sql

SELECT
  CASE
    WHEN a.id IS NOT NULL AND b.id IS NOT NULL THEN 'in_both'
    WHEN a.id IS NOT NULL THEN 'only_in_docling'
    ELSE 'only_in_arxiv_tex_extract'
  END AS status,
  COUNT(*) AS count
FROM (SELECT DISTINCT arxiv_id AS id FROM docling) a
FULL OUTER JOIN (SELECT DISTINCT arxiv_id AS id FROM arxiv_tex_extract) b
  ON a.id = b.id
GROUP BY 1
ORDER BY 1;

Running query in 'duckdb:///:memory:'

status,count
in_both,2705


In [10]:
%%sql

SELECT
  x.id,
  x.presence_status,
  d.status AS docling_status,
  a.status AS arxiv_tex_extract_status
FROM (
    SELECT
      COALESCE(a.id, b.id) AS id,
      CASE
        WHEN a.id IS NOT NULL AND b.id IS NOT NULL THEN 'in_both'
        WHEN a.id IS NOT NULL THEN 'only_in_docling'
        ELSE 'only_in_arxiv_tex_extract'
      END AS presence_status
    FROM (SELECT DISTINCT arxiv_id AS id FROM docling) a
    FULL OUTER JOIN (SELECT DISTINCT arxiv_id AS id FROM arxiv_tex_extract) b
      ON a.id = b.id
) x
LEFT JOIN docling d
  ON x.id = d.arxiv_id
LEFT JOIN arxiv_tex_extract a
  ON x.id = a.arxiv_id
WHERE x.presence_status IN ('only_in_docling', 'only_in_arxiv_tex_extract')
ORDER BY x.id;

Running query in 'duckdb:///:memory:'

id,presence_status,docling_status,arxiv_tex_extract_status
